In [78]:
import numpy as np

def generate_utility_functions(
    num_players=6, subissues_per_issue=[3, 3, 4, 4, 5], max_utility_per_player=100, subissue_budget=10, threshold_strategy="proportional"
):
    """
    Generate Pareto-optimal utility functions with intelligent thresholds.

    Args:
        num_players (int): Number of players.
        subissues_per_issue (list): List specifying the number of subissues for each issue.
        max_utility_per_player (int): Maximum utility per player.
        subissue_budget (int): Fixed total utility budget per subissue. Used to enforce pareto optemality?
        threshold_strategy (str): Strategy for assigning thresholds ("proportional", "balanced", "competitive", "random").

    Returns:
        utilities (list of np.ndarray): Utility matrices for all players.
        thresholds (list): Threshold utility values for each player.
    """
    num_issues = len(subissues_per_issue)
    max_subissues = max(subissues_per_issue)
    
    utilities = []
    total_utilities = []
    
    # Initialize utility matrices for each player
    for _ in range(num_players):
        utilities.append(np.negative(np.ones((num_issues, max_subissues), dtype=int)))
    
    # Allocate utilities for each subissue
    for i, num_subissues in enumerate(subissues_per_issue):
        for j in range(num_subissues):
            # Start with a random allocation that sums to subissue_budget
            allocation = np.random.multinomial(subissue_budget, np.ones(num_players) / num_players)
            
            # Assign the allocation to all players for the current subissue
            for player in range(num_players):
                utilities[player][i, j] = allocation[player]
    
    # Compute total utilities for each player
    for player in range(num_players):
        total_utilities.append(utilities[player].sum())
    
    # Enforce maximum utility per player
    for player in range(num_players):
        total_utility = total_utilities[player]
        if total_utility > max_utility_per_player:
            scale_factor = max_utility_per_player / total_utility
            utilities[player] = (utilities[player] * scale_factor).astype(int)
            total_utilities[player] = utilities[player].sum()
    
    # Calculate thresholds
    thresholds = []
    if threshold_strategy == "proportional":
        for total in total_utilities:
            thresholds.append(int(total * np.random.uniform(0.6, 0.9)))  # Proportional to total utility
    elif threshold_strategy == "balanced":
        total_available_utility = sum(total_utilities)
        target_threshold_sum = total_available_utility * 0.8  # 80% of total available utility
        for total in total_utilities:
            thresholds.append(int(total * (target_threshold_sum / total_available_utility)))
    elif threshold_strategy == "competitive":
        ranked_utilities = sorted(total_utilities, reverse=True)
        for total in total_utilities:
            rank = ranked_utilities.index(total) + 1
            thresholds.append(int(total * (1 - (rank / num_players))))  # Competitive adjustment
    elif threshold_strategy == "random":
        for total in total_utilities:
            thresholds.append(np.random.randint(int(total * 0.6), int(total * 0.9)))
    
    return utilities, thresholds

# Example usage
subissues = [3, 3, 4, 4, 5]  # User-defined number of subissues per issue
utilities, thresholds = generate_utility_functions(
    num_players=6, subissues_per_issue=subissues, max_utility_per_player=100, subissue_budget=10, threshold_strategy="competitive"
)

for i, u in enumerate(utilities):
    print(f"Player {i+1} Utility Matrix:\n{u}\n")
print(f"Thresholds: {thresholds}")


Player 1 Utility Matrix:
[[ 2  1  1 -1 -1]
 [ 2  2  3 -1 -1]
 [ 2  2  3  3 -1]
 [ 1  0  3  1 -1]
 [ 1  2  1  2  1]]

Player 2 Utility Matrix:
[[ 1  0  1 -1 -1]
 [ 0  2  1 -1 -1]
 [ 3  2  1  1 -1]
 [ 1  1  2  1 -1]
 [ 3  2  2  1  3]]

Player 3 Utility Matrix:
[[ 2  5  2 -1 -1]
 [ 2  0  3 -1 -1]
 [ 1  1  1  3 -1]
 [ 2  2  1  1 -1]
 [ 0  0  3  1  2]]

Player 4 Utility Matrix:
[[ 3  3  4 -1 -1]
 [ 3  1  1 -1 -1]
 [ 1  1  0  1 -1]
 [ 3  2  2  4 -1]
 [ 1  3  2  2  0]]

Player 5 Utility Matrix:
[[ 1  0  1 -1 -1]
 [ 3  3  0 -1 -1]
 [ 1  2  0  0 -1]
 [ 1  4  2  0 -1]
 [ 3  2  1  2  3]]

Player 6 Utility Matrix:
[[ 1  1  1 -1 -1]
 [ 0  2  2 -1 -1]
 [ 2  2  5  2 -1]
 [ 2  1  0  3 -1]
 [ 2  1  1  2  1]]

Thresholds: [18, 0, 13, 25, 3, 8]
